#### Step 1: Go from words to numbers

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

x = tokenizer("40°F in Fayetteville", return_tensors="pt")
print(x)
for i in x["input_ids"][0]:
    print(i, tokenizer.decode(i))


{'input_ids': tensor([[  101,  2871,  7737,  2546,  1999, 19243,  4674,  3077,   102]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}
tensor(101) [CLS]
tensor(2871) 40
tensor(7737) ##°
tensor(2546) ##f
tensor(1999) in
tensor(19243) faye
tensor(4674) ##tte
tensor(3077) ##ville
tensor(102) [SEP]


In [2]:
for i in range(2000,2015):
    print(i, tokenizer.decode(i))

2000 to
2001 was
2002 he
2003 is
2004 as
2005 for
2006 on
2007 with
2008 that
2009 it
2010 his
2011 by
2012 at
2013 from
2014 her


In [3]:
tokenizer.vocab_size

30522

In [4]:
bert_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)
bert_model

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [5]:
bert_model.distilbert.embeddings.word_embeddings

Embedding(30522, 768, padding_idx=0)

In [6]:
import torch

f = bert_model.distilbert.embeddings.word_embeddings
t = torch.tensor(1000, dtype=torch.int64)
f(t)

tensor([ 3.8553e-03, -2.0628e-02, -4.4224e-02, -7.1081e-04,  4.0089e-02,
        -2.9143e-02, -3.5146e-02, -1.3172e-02,  1.4067e-02, -3.9761e-03,
        -4.7754e-02,  3.7087e-03, -1.6355e-02,  3.5946e-02, -2.1759e-02,
        -4.8425e-02, -4.3623e-03, -3.8150e-02,  2.4604e-02,  1.8415e-02,
        -2.2256e-02, -3.4410e-02, -2.8910e-03, -6.4880e-03, -9.4455e-03,
        -2.7478e-02, -1.0296e-02, -1.2149e-02,  1.3651e-02, -1.4377e-02,
        -2.8772e-02,  3.9488e-02, -2.1691e-03,  1.2043e-02,  2.1120e-02,
         5.7697e-03, -1.2646e-03, -1.7215e-02, -2.7267e-02, -1.3637e-02,
         1.5574e-02, -4.8309e-03, -3.2222e-02, -8.0461e-03, -4.0717e-02,
        -1.8014e-02,  1.0456e-02,  8.5044e-03, -2.7720e-02,  1.9996e-02,
        -4.5963e-02, -7.4484e-03,  2.0851e-03, -5.4291e-02,  1.1432e-02,
        -1.0152e-02,  1.2473e-03,  1.2779e-02,  6.7817e-03,  2.3927e-02,
         3.0850e-03, -4.5921e-02,  4.4618e-03, -4.1673e-02,  2.7283e-03,
         4.1739e-03,  1.6673e-02, -1.3203e-02,  2.1

#### Step 2: Make a simple model

In [7]:
import torch
import torch.nn as nn

class SequenceAveragingModel(nn.Module):
    def __init__(self):
        super().__init__()
        embedding_dim = 768
        output_dim = 2
        self.embedding_module = bert_model.distilbert.embeddings.word_embeddings
        self.a_linear_module = nn.Linear(embedding_dim, output_dim)

    def forward(self, x, attention_mask):
        embeddings = self.embedding_module(x)
        averaged_embeddings = torch.mean(embeddings, dim=1)
        output = self.a_linear_module(averaged_embeddings)
        return output

model = SequenceAveragingModel()
model

SequenceAveragingModel(
  (embedding_module): Embedding(30522, 768, padding_idx=0)
  (a_linear_module): Linear(in_features=768, out_features=2, bias=True)
)

In [8]:
for name, p in model.named_parameters():
    print(f"{name:30s} {p.numel():>12,}")

embedding_module.weight          23,440,896
a_linear_module.weight                1,536
a_linear_module.bias                      2


In [9]:
inputs = tokenizer("40°F in Fayetteville", return_tensors="pt")
model(inputs["input_ids"], attention_mask=inputs["attention_mask"])


tensor([[-0.0075, -0.0316]], grad_fn=<AddmmBackward0>)